# Exploration des facteurs d'émission (ADEME, Base Carbone)

Source : API Base Carbone de l'ADEME (data.ademe.fr).
Facteurs retenus : avion (43743), TGV (43256), Intercités (43272), TER (43255), voiture (43791).
Avion : 101 à 220 sièges, vols de 500 à 1 000 km, avec traînées (un seul facteur pour toutes
les liaisons : limite à signaler).

In [1]:
import os
import sys

# on se place à la racine du projet
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.append(os.getcwd())

from src.collecte import get_base_carbone

url = "https://data.ademe.fr/data-fair/api/v1/datasets/base-carboner/lines"
bc = get_base_carbone(url, ["43743", "43256", "43272", "43255", "43791"])

# les colonnes qui décrivent chaque facteur, une ligne par facteur
colonnes = ["Identifiant_de_l'élément", "Nom_base_français", "Nom_attribut_français",
            "Nom_frontière_français", "Total_poste_non_décomposé", "Unité_français",
            "Statut_de_l'élément", "Localisation_géographique", "Date_de_modification"]
bc[colonnes]

,Identifiant_de_l'élément,Nom_base_français,Nom_attribut_français,Nom_frontière_français,Total_poste_non_décomposé,Unité_français,Statut_de_l'élément,Localisation_géographique,Date_de_modification
0,43743,Avion passagers,"101-220 sièges, 500-1000 kms, 2023",AVEC trainées,0.22500,kgCO2e/passager.km,Valide générique,France continentale,2025-04-24
1,43272,Intercités,2022,NaN,0.00750,kgCO2e/passager.km,Valide générique,France continentale,2024-07-04
2,43255,TER,2022,Traction moyenne,0.02770,kgCO2e/passager.km,Valide générique,France continentale,2023-09-27
3,43256,TGV,2022,NaN,0.00293,kgCO2e/passager.km,Valide générique,France continentale,2023-09-27
4,43791,Voiture,Motorisation moyenne,2023,0.25600,kgCO2e/km,Valide générique,France continentale,2025-04-24


## Constats

On utilise 5 facteurs d'émission carbone, qui correspond à chaque mode de transport, sur 49 colonnes (décrivant le gaz, la qualité de la données). Ces 49 colonnes vont être simplifiées pour ne pas compléxifier inutilement l'analyse centrale. Tous les facteurs ont ici le statut "Valide générique", ce qui suit les recommandations de l'ADEME.
Il y a une attention particulière qui doit être portée sur les unités, elles ne sont pas homogènes entre elles. Pour l'avion le TGV, les Intercités et le TER, l'unité de référence est le kgCO2e par passager.km 
La voiture utilise le kgCO2e oar véhicule.km. POur les comparer, il faut diviser par le nombre d'occupants par voiture. Il s'agit d'une hypothèse supplémentaire. Il y a un problème d'ajustement au niveau des années aussi. Le train c'est 2022, pour l'avion et la voiture c'est 2023. Il s'agit des dernières valeurs publiées par l'ADEME. Ce léger décalage ne change pas drastiquement les interprétations et les résultats à observer, car il s'agit surtout de facteurs technologiques et tendanciels. Ils évoluent donc lentement. I reste tout de même essentiel de mentionner ce décalage. 

## Ce que fera le nettoyage (`clean_base_carbone`)

1. Garder une ligne par mode, qui sera composée du mode, de l'identifiant ADEME, de la valeur, de l'unité ;
2. Donner des noms plus courts aux valeurs de la variable mode : `avion`, `tgv`, `intercites`, `ter`, `voiture` ;
3. vérifier par des `assert`qu'il y a bien un seul facteur par mode et aucune valeur manquante.

In [2]:
from src.nettoyage import clean_base_carbone

modes = {"43743": "avion", "43256": "tgv", "43272": "intercites", "43255": "ter", "43791": "voiture"}
facteurs = clean_base_carbone(bc, modes)
facteurs

,mode,code_ademe,co2_kg_km,unite
0,avion,43743,0.22500,kgCO2e/passager.km
1,intercites,43272,0.00750,kgCO2e/passager.km
2,ter,43255,0.02770,kgCO2e/passager.km
3,tgv,43256,0.00293,kgCO2e/passager.km
4,voiture,43791,0.25600,kgCO2e/km


In [3]:
# sauvegarde de la table propre
facteurs.to_parquet("data/clean/facteurs_emission.parquet")
print("modes :", facteurs["mode"].tolist())

modes : ['avion', 'intercites', 'ter', 'tgv', 'voiture']


## Bilan

Fichier propre : `data/clean/facteurs_emission.parquet`

**Limites:**
En raison d'une contrainte de temps, on ne considère qu'un facteur pour un avion pour les liaisons, alors que les vols sont de distance plus courtes que les autres. Or, un avion qui parcours des distances plus courtes consomme plus au km. Les mêmes facteurs sont appliqués en 2017 et en 2023. Il s'agit de comparer à technologie égale. Cela peut être une limite, si on cherche à comparer les émissions sur le long terme et à comptabiliser leur diminution notamment. 